# EXP-002B — EXP-001 seed sweep

Analysis notebook. Runs the EXP-001 visible-pixel random policy across multiple seeds to measure stochastic variance and find a better candidate seed.

Do not submit this notebook as the scoring notebook. If a seed wins locally, create a separate scoring notebook pinned to that seed.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, random, zlib
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

EXP_ID='EXP-002B'
MAX_MOVES=1000
SEEDS=[0,1,2,3,4,5,10,42,100,123]
USE_PER_GAME_SEED=False
WORK=Path('/kaggle/working')
ART=WORK/'exp002b_seed_sweep_artifacts'
ART.mkdir(exist_ok=True)
envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'
(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=offline\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)

def rng_for(g, seed):
    return random.Random(seed if not USE_PER_GAME_SEED else seed+(zlib.adler32(g.encode())&0xffffffff))

def pix(frame,rng):
    color=rng.choice(np.unique(frame).tolist())
    ys,xs=np.where(frame==color)
    i=rng.randint(0,len(xs)-1)
    return {'x':int(xs[i]),'y':int(ys[i])}

def play(env,g,seed):
    rng=rng_for(g,seed)
    r=env._last_response
    last=None
    action_counts=defaultdict(int)
    # Exact EXP-001 policy: include every non-RESET GameAction, including ACTION7 if enum has it.
    acts=[a for a in GameAction if a is not GameAction.RESET]
    for m in range(1,MAX_MOVES+1):
        if r is None or r.state==GameState.WIN:
            break
        if r.state in (GameState.GAME_OVER,GameState.NOT_PLAYED):
            r=env.step(GameAction.RESET,{})
            last='RESET'
            action_counts['RESET']+=1
            continue
        a=rng.choice(acts)
        data=pix(r.frame[-1],rng) if a.is_complex() and r.frame else {}
        r=env.step(a,data,reasoning={'exp_id':EXP_ID,'seed':seed,'policy':'exp001_seed_sweep'})
        last=a.name
        action_counts[a.name]+=1
    return {'game_id':g,'moves':m,'state':r.state.name if r else 'unknown','levels_completed':int(r.levels_completed) if r else 0,'last_action':last,'action_counts':dict(action_counts)}

def run_seed(seed):
    arcade=arc_agi.Arcade()
    infos=list(arcade.available_environments)
    rows=[]; details=[]
    print(f'Running seed={seed} envs={len(infos)}')
    for i,info in enumerate(infos,1):
        row=play(arcade.make(info.game_id),info.game_id,seed)
        details.append(row)
        flat={k:v for k,v in row.items() if k!='action_counts'}
        rows.append(flat)
        print(f'  [{i:03d}/{len(infos):03d}] {flat}')
    sc=arcade.get_scorecard()
    scorecard_rows=[{'seed':seed,'game':e.id,'score':float(e.score),'levels_completed':int(e.levels_completed),'actions':int(e.actions),'completed':bool(e.completed)} for e in sc.environments]
    tag_rows=[{'seed':seed,'tag':t.id,'score':float(t.score),'levels_completed':int(t.levels_completed),'number_of_environments':int(t.number_of_environments)} for t in (sc.tags_scores or [])]
    summary={'seed':seed,'score':float(sc.score),'total_environments_completed':int(sc.total_environments_completed),'total_environments':int(sc.total_environments),'total_levels_completed':int(sc.total_levels_completed),'total_levels':int(sc.total_levels),'total_actions':int(sc.total_actions)}
    return summary, rows, details, scorecard_rows, tag_rows

summaries=[]; all_games=[]; all_tags=[]; all_details={}
for seed in SEEDS:
    summary, rows, details, scorecard_rows, tag_rows = run_seed(seed)
    summaries.append(summary)
    all_games.extend(scorecard_rows)
    all_tags.extend(tag_rows)
    all_details[str(seed)]={'run_rows':rows,'details':details,'summary':summary}
    print('SEED_RESULT', summary)

summary_df=pd.DataFrame(summaries).sort_values(['score','total_levels_completed'],ascending=False)
summary_df.to_csv(ART/'exp002b_seed_sweep_summary.csv',index=False)
games_df=pd.DataFrame(all_games)
tags_df=pd.DataFrame(all_tags)
games_df.to_csv(ART/'exp002b_seed_sweep_by_environment.csv',index=False)
tags_df.to_csv(ART/'exp002b_seed_sweep_by_tag.csv',index=False)
(ART/'exp002b_seed_sweep_details.json').write_text(json.dumps(all_details,indent=2))

best=summary_df.iloc[0].to_dict()
best_seed=int(best['seed'])
(ART/'exp002b_best_seed_summary.json').write_text(json.dumps(best,indent=2))
games_df[games_df['seed'].astype(int)==best_seed].to_csv(ART/'exp002b_best_seed_per_game.csv',index=False)
tags_df[tags_df['seed'].astype(int)==best_seed].to_csv(ART/'exp002b_best_seed_by_tag.csv',index=False)

sp=WORK/'submission.parquet'
if not sp.exists():
    pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
manifest=sorted(str(p) for p in ART.glob('*'))
pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('Best seed:', best)
print(summary_df)
summary_df
